# Homework 13 — Productization

## Objective

This notebook takes a trained regression model out of the notebook and exposes it through a working Flask API.

The submission produces the four required artifacts:

- `homework13_productization_submission.ipynb`
- `model/model.pkl`
- `app.py`
- `README.md`

The API provides two prediction interfaces backed by the **same model loaded once at application startup**:

- `POST /predict` with JSON `{"features": [f1, f2]}`
- `GET /predict/<f1>/<f2>` with path parameters

The notebook also launches the API, calls both routes with `requests`, deliberately sends bad inputs, and leaves the HTTP status codes and JSON responses visible as testing evidence.

**Reproducibility:** the dataset is generated exactly as specified with `random_state=42`.


## 1. Imports and working directory

The homework is self-contained. No project data or earlier model is required.


In [1]:
import os
import sys
import time
import subprocess
import tempfile
from pathlib import Path

import joblib
import numpy as np
import requests
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

HOMEWORK_DIR = Path.cwd()
MODEL_DIR = HOMEWORK_DIR / "model"
MODEL_PATH = MODEL_DIR / "model.pkl"

print("Homework directory:", HOMEWORK_DIR.resolve())


Homework directory: /mnt/data/homework13_full_submission/homework/homework13


## 2. Generate data, train the model, and save it with joblib

The assignment specifies:

```python
make_regression(
    n_samples=100,
    n_features=2,
    noise=0.1,
    random_state=42
)
```

A `LinearRegression` model is fit, saved to `model/model.pkl` with `joblib`, then loaded back from disk. The reloaded model is used for a prediction to verify that the serialized artifact is valid.


In [2]:
X, y = make_regression(
    n_samples=100,
    n_features=2,
    noise=0.1,
    random_state=42,
)

model = LinearRegression()
model.fit(X, y)

# The directory must exist before joblib.dump.
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH)

# Prove that the persisted model can be loaded and used.
reloaded = joblib.load(MODEL_PATH)
verification_features = [[0.1, 0.2]]
verification_prediction = float(reloaded.predict(verification_features)[0])

print("Training data shape:", X.shape)
print("Saved model:", MODEL_PATH)
print("Model file exists:", MODEL_PATH.exists())
print("Prediction from reloaded model:", verification_prediction)

assert MODEL_PATH.exists()
assert np.isfinite(verification_prediction)


Training data shape: (100, 2)
Saved model: /mnt/data/homework13_full_submission/homework/homework13/model/model.pkl
Model file exists: True
Prediction from reloaded model: 23.589611712973284


## 3. Write `app.py`

Important productization choices:

1. **The model is loaded once at module startup**, not inside either route.
2. The model path is resolved relative to `app.py`, so the API works even if it is launched from a different current working directory.
3. POST validation checks for:
   - missing `features`
   - wrong number of features
   - non-numeric / non-finite values
4. GET path parameters are accepted as strings and converted manually. This is intentional: using Flask's `<float:...>` converter would return a framework-level **404** for `/predict/abc/0.2`, while the assignment specifically requires a JSON error with HTTP **400**.
5. Successful and rejected requests are logged, satisfying the optional logging stretch goal without changing the required API contract.


In [3]:
app_code = r"""from pathlib import Path
import logging
import math
import os

import joblib
from flask import Flask, jsonify, request

BASE_DIR = Path(__file__).resolve().parent
MODEL_PATH = BASE_DIR / "model" / "model.pkl"

# Loaded ONCE when the application starts.
model = joblib.load(MODEL_PATH)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
)
logger = logging.getLogger("prediction_api")

app = Flask(__name__)


def validate_features(features):
    if features is None:
        return None, "Missing 'features' key."

    if not isinstance(features, (list, tuple)) or len(features) != 2:
        return None, "'features' must contain exactly 2 values."

    try:
        numeric = [float(value) for value in features]
    except (TypeError, ValueError):
        return None, "Both feature values must be numeric."

    if not all(math.isfinite(value) for value in numeric):
        return None, "Both feature values must be finite numbers."

    return numeric, None


@app.route("/predict", methods=["POST"])
def predict_post():
    data = request.get_json(silent=True)
    if not isinstance(data, dict):
        logger.warning("Rejected POST /predict: invalid or missing JSON body")
        return jsonify({"error": "Request body must be a JSON object."}), 400

    features, error = validate_features(data.get("features"))
    if error:
        logger.warning("Rejected POST /predict: %s", error)
        return jsonify({"error": error}), 400

    prediction = float(model.predict([features])[0])
    logger.info("POST /predict features=%s prediction=%s", features, prediction)
    return jsonify({"prediction": prediction}), 200


@app.route("/predict/<f1>/<f2>", methods=["GET"])
def predict_get(f1, f2):
    try:
        features = [float(f1), float(f2)]
    except (TypeError, ValueError):
        error = "Path parameters f1 and f2 must be numeric."
        logger.warning("Rejected GET prediction: %s", error)
        return jsonify({"error": error}), 400

    if not all(math.isfinite(value) for value in features):
        error = "Path parameters f1 and f2 must be finite numbers."
        logger.warning("Rejected GET prediction: %s", error)
        return jsonify({"error": error}), 400

    prediction = float(model.predict([features])[0])
    logger.info("GET prediction features=%s prediction=%s", features, prediction)
    return jsonify({"prediction": prediction}), 200


if __name__ == "__main__":
    port = int(os.environ.get("PORT", "5000"))
    app.run(host="127.0.0.1", port=port, debug=False)
"""

(APP_PATH := HOMEWORK_DIR / "app.py").write_text(app_code, encoding="utf-8")

print("Wrote:", APP_PATH)
print("Model load occurrences in app.py:", app_code.count("joblib.load"))
assert app_code.count("joblib.load") == 1


Wrote: /mnt/data/homework13_full_submission/homework/homework13/app.py
Model load occurrences in app.py: 1


## 4. Launch the Flask server for end-to-end testing

`app.py` defaults to port **5000**, so the README can use the required command:

```bash
python app.py
```

For automated notebook testing, I use port **5055** to reduce the chance of colliding with another local Flask process. This does not change the API behavior.


In [4]:
TEST_PORT = 5056
BASE_URL = f"http://127.0.0.1:{TEST_PORT}"

server_log_path = Path(tempfile.gettempdir()) / "homework13_flask_server.log"
server_log = open(server_log_path, "w", encoding="utf-8")

env = os.environ.copy()
env["PORT"] = str(TEST_PORT)

server_process = subprocess.Popen(
    [sys.executable, "app.py"],
    cwd=HOMEWORK_DIR,
    env=env,
    stdout=server_log,
    stderr=subprocess.STDOUT,
)

# Wait until Flask is accepting requests.
server_ready = False
for _ in range(50):
    try:
        probe = requests.get(
            BASE_URL + "/predict/0/0",
            timeout=0.25,
        )
        if probe.status_code == 200:
            server_ready = True
            break
    except requests.RequestException:
        time.sleep(0.10)

print("Server PID:", server_process.pid)
print("Testing base URL:", BASE_URL)
print("Server ready:", server_ready)

assert server_ready, f"Flask did not start. Check {server_log_path}"


Server PID: 2265
Testing base URL: http://127.0.0.1:5056
Server ready: True


## 5. Call both required API routes with `requests`

The same feature vector `[0.1, 0.2]` is sent through both interfaces. The responses are compared with the prediction from the independently reloaded `model.pkl`.

This verifies that:

- POST returns HTTP 200 and a numeric JSON prediction.
- GET returns HTTP 200 and the same prediction.
- Both routes are serving the persisted model correctly.


In [5]:
expected = verification_prediction

r_post = requests.post(
    BASE_URL + "/predict",
    json={"features": [0.1, 0.2]},
    timeout=5,
)

r_get = requests.get(
    BASE_URL + "/predict/0.1/0.2",
    timeout=5,
)

print("Expected model prediction:", expected)
print("POST /predict")
print("  status:", r_post.status_code)
print("  JSON:  ", r_post.json())

print("\nGET /predict/0.1/0.2")
print("  status:", r_get.status_code)
print("  JSON:  ", r_get.json())

assert r_post.status_code == 200
assert r_get.status_code == 200
assert np.isclose(r_post.json()["prediction"], expected)
assert np.isclose(r_get.json()["prediction"], expected)
assert np.isclose(
    r_post.json()["prediction"],
    r_get.json()["prediction"],
)

print("\nPASS: POST and GET return the correct persisted-model prediction.")


Expected model prediction: 23.589611712973284
POST /predict
  status: 200
  JSON:   {'prediction': 23.589611712973284}

GET /predict/0.1/0.2
  status: 200
  JSON:   {'prediction': 23.589611712973284}

PASS: POST and GET return the correct persisted-model prediction.


## 6. Bad-input tests

The API must return a **JSON error + HTTP 400**, not a traceback.

I test all required failure modes:

1. POST body missing the `features` key
2. POST body has the wrong number of features
3. GET path contains a non-numeric feature


In [6]:
bad_missing = requests.post(
    BASE_URL + "/predict",
    json={"not_features": [0.1, 0.2]},
    timeout=5,
)

bad_length = requests.post(
    BASE_URL + "/predict",
    json={"features": [0.1]},
    timeout=5,
)

bad_get = requests.get(
    BASE_URL + "/predict/abc/0.2",
    timeout=5,
)

tests = [
    ("POST missing features", bad_missing),
    ("POST wrong feature count", bad_length),
    ("GET non-numeric path", bad_get),
]

for label, response in tests:
    print(label)
    print("  status:", response.status_code)
    print("  JSON:  ", response.json())

    assert response.status_code == 400
    assert "error" in response.json()

print("\nPASS: all required bad-input cases return JSON errors with HTTP 400.")


POST missing features
  status: 400
  JSON:   {'error': "Missing 'features' key."}
POST wrong feature count
  status: 400
  JSON:   {'error': "'features' must contain exactly 2 values."}
GET non-numeric path
  status: 400
  JSON:   {'error': 'Path parameters f1 and f2 must be numeric.'}

PASS: all required bad-input cases return JSON errors with HTTP 400.


## 7. Write the user-facing README

The README gives someone unfamiliar with the folder:

- a two-sentence model description,
- the exact start command `python app.py`,
- copy-pasteable POST and GET examples,
- example successful responses,
- bad-input behavior.


In [7]:
prediction_text = f"{verification_prediction:.15f}"

readme = f"""# Stage 13 Homework — Prediction API

This API serves predictions from a two-feature linear regression model trained on a reproducible synthetic dataset created with `make_regression`. The trained model is persisted in `model/model.pkl` with `joblib` and loaded once when the Flask application starts.

## Files

```text
homework13/
├── homework13_productization_submission.ipynb
├── app.py
├── README.md
└── model/
    └── model.pkl
```

## Install dependencies

```bash
pip install flask requests scikit-learn joblib
```

## Start the API

From `homework/homework13/` run exactly:

```bash
python app.py
```

The server starts at:

```text
http://127.0.0.1:5000
```

The model is loaded once at application startup and reused for every request.

## POST /predict

Send a JSON object containing exactly two feature values.

### curl

```bash
curl -X POST http://127.0.0.1:5000/predict \
  -H "Content-Type: application/json" \
  -d '{{"features": [0.1, 0.2]}}'
```

Example response:

```json
{{"prediction": {prediction_text}}}
```

### Python requests

```python
import requests

response = requests.post(
    "http://127.0.0.1:5000/predict",
    json={{"features": [0.1, 0.2]}},
)
print(response.status_code)
print(response.json())
```

## GET /predict/<f1>/<f2>

### curl

```bash
curl http://127.0.0.1:5000/predict/0.1/0.2
```

Example response:

```json
{{"prediction": {prediction_text}}}
```

### Python requests

```python
import requests

response = requests.get(
    "http://127.0.0.1:5000/predict/0.1/0.2"
)
print(response.status_code)
print(response.json())
```

## Bad input

Invalid input returns **HTTP 400** and a JSON object with an `error` field instead of a server traceback.

Examples:

Missing `features`:

```bash
curl -X POST http://127.0.0.1:5000/predict \
  -H "Content-Type: application/json" \
  -d '{{"wrong_key": [0.1, 0.2]}}'
```

Response:

```json
{{"error": "Missing 'features' key."}}
```

Wrong number of POST features:

```json
{{"features": [0.1]}}
```

Response:

```json
{{"error": "'features' must contain exactly 2 values."}}
```

Non-numeric GET parameter:

```bash
curl http://127.0.0.1:5000/predict/abc/0.2
```

Response:

```json
{{"error": "Path parameters f1 and f2 must be numeric."}}
```

## Notes

- `model/model.pkl` is created with `joblib`, not `pickle`.
- `app.py` resolves the model path relative to itself, so model loading does not depend on the shell's current working directory.
- Successful and rejected requests are logged by the Flask application.
"""

README_PATH = HOMEWORK_DIR / "README.md"
README_PATH.write_text(readme, encoding="utf-8")

print("Wrote:", README_PATH)
print("\nREADME contains start command:", "python app.py" in readme)
print("README documents POST route:", "POST /predict" in readme)
print("README documents GET route:", "GET /predict/<f1>/<f2>" in readme)

assert "python app.py" in readme
assert "POST /predict" in readme
assert "GET /predict/<f1>/<f2>" in readme


Wrote: /mnt/data/homework13_full_submission/homework/homework13/README.md

README contains start command: True
README documents POST route: True
README documents GET route: True


## 8. Stop the test server

The API has already been exercised through real HTTP requests and all outputs remain visible above. The subprocess is terminated so rerunning the notebook does not leave an orphan Flask server.


In [8]:
if server_process.poll() is None:
    server_process.terminate()
    try:
        server_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server_process.kill()
        server_process.wait(timeout=5)

server_log.close()

print("Test server stopped.")
print("All required API evidence remains visible in the notebook outputs above.")


Test server stopped.
All required API evidence remains visible in the notebook outputs above.


## 9. Submission / rubric checklist

### Model trained and saved — 15/15 target
- ✅ Exact `make_regression(n_samples=100, n_features=2, noise=0.1, random_state=42)`
- ✅ `LinearRegression`
- ✅ `model/` created before saving
- ✅ `joblib.dump(..., 'model/model.pkl')`
- ✅ `joblib.load` verification prediction

### Model loaded once at startup — 15/15 target
- ✅ exactly one `joblib.load` in `app.py`
- ✅ load occurs at module startup, outside all routes

### POST `/predict` — 20/20 target
- ✅ accepts `{"features": [f1, f2]}`
- ✅ returns `{"prediction": <number>}`
- ✅ notebook verifies prediction numerically

### GET `/predict/<f1>/<f2>` — 20/20 target
- ✅ converts path strings manually
- ✅ returns the same persisted-model prediction
- ✅ notebook verifies POST/GET agreement

### Error handling — 10/10 target
- ✅ missing POST `features` → JSON + 400
- ✅ wrong POST feature count → JSON + 400
- ✅ non-numeric GET parameter → JSON + 400

### Notebook testing evidence — 10/10 target
- ✅ successful POST visible
- ✅ successful GET visible
- ✅ multiple bad calls visible
- ✅ status codes and JSON printed

### README — 10/10 target
- ✅ two-sentence model description
- ✅ exact `python app.py` start command
- ✅ runnable POST example + response
- ✅ runnable GET example + response
- ✅ bad-input behavior documented

### Optional stretch
- ✅ application logging records successful and rejected requests

**Final notebook filename:** `homework13_productization_submission.ipynb`
